In [6]:
import os
import sys
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib            import Path
from collections        import defaultdict
from torch.utils.data   import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics    import balanced_accuracy_score, accuracy_score
from sklearn.inspection import permutation_importance
from sklearn.base       import BaseEstimator, ClassifierMixin

sys.path.append('..')
import module

from src.model.modules   import (MicroExpressionDataset, 
                                 Normalizer, 
                                 SpatialTemporalCNN, 
                                 FeatureExtractor)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
print(f'Device: {device}')

Device: cuda


In [7]:
# ── Load dataset & GroupKFold ─────────────────────────────────────────────
ANNOTATIONS = os.path.join(Path.home(), "datasets", "anxiety_raw", "annotations-v2.csv")

dataset = MicroExpressionDataset(annotations_file=ANNOTATIONS, 
                                 stage='all',
                                 mode='train', 
                                 valid_only=True)

folds = MicroExpressionDataset.build_group_kfold(dataset.annotations, n_splits=5)

print(f'Dataset: {len(dataset)} samples, {dataset.annotations["subject_id"].nunique()} subjek')

Dataset: 559 samples, 56 subjek


In [8]:
# ── Re-train Strategi C pada semua fold dan kumpulkan prediksi per-clip ───
# Tujuan: dapatkan prediksi probabilitas tiap clip untuk analisis detail

EPOCHS       = 100
LR           = 3e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 32
PATIENCE     = 20

# Hasil dikumpulkan: satu baris per clip
all_records = []

for fold_info in folds:
    fold      = fold_info['fold']
    train_idx = fold_info['train_idx']
    test_idx  = fold_info['test_idx']

    # Normalisasi
    dataset.set_mode('eval'); dataset.clear_cache()
    X_all, y_all, subs_all = dataset.get_all_features()
    X_tr, y_tr = X_all[train_idx], y_all[train_idx]
    X_te, y_te = X_all[test_idx],  y_all[test_idx]
    norm = Normalizer()
    X_tr = norm.fit_transform(X_tr)
    X_te = norm.transform(X_te)

    subs_test = [subs_all[i] for i in test_idx]
    ann_test  = dataset.annotations.iloc[test_idx].reset_index(drop=True)

    counts  = np.bincount(y_tr, minlength=2)
    weights = torch.tensor([1./counts[l] for l in y_tr], dtype=torch.float)
    sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
    tr_ds   = TensorDataset(torch.from_numpy(X_tr),
                             torch.from_numpy(y_tr).long(),
                             torch.arange(len(X_tr)))
    te_ds   = TensorDataset(torch.from_numpy(X_te),
                             torch.from_numpy(y_te).long(),
                             torch.arange(len(X_te)))
    tr_ld   = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True)
    te_ld   = DataLoader(te_ds, batch_size=BATCH_SIZE*2, shuffle=False)

    model     = SpatialTemporalCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler('cuda')
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    best_loss = float('inf'); best_uar = -1.; best_ckpt = None; no_imp = 0
    dataset.set_mode('train')

    for epoch in range(EPOCHS):
        model.train()
        for feat, label, _ in tr_ld:
            feat, label = feat.to(device), label.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                loss = criterion(model(feat), label)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        scheduler.step()

        # Eval ringan untuk ckpt
        model.eval()
        with torch.no_grad():
            probs_ep, true_ep = [], []
            for feat, label, _ in te_ld:
                p = torch.softmax(model(feat.to(device)),dim=1).cpu().numpy()
                probs_ep.extend(np.argmax(p,axis=1).tolist())
                true_ep.extend(label.numpy().tolist())
        uar_ep = balanced_accuracy_score(true_ep, probs_ep)
        if uar_ep > best_uar:
            best_uar  = uar_ep
            best_ckpt = {k:v.cpu().clone() for k,v in model.state_dict().items()}

        # Train loss tracking
        tr_loss = 0.
        model.train()
        with torch.no_grad():
            for feat, label, _ in tr_ld:
                out = model(feat.to(device))
                tr_loss += criterion(out, label.to(device)).item()
        tr_loss /= max(len(tr_ld), 1)
        if tr_loss < best_loss - 1e-4:
            best_loss = tr_loss; no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE: break

    model.load_state_dict(best_ckpt)

    # Kumpulkan prediksi per clip dengan metadata lengkap
    model.eval()
    all_probs = []
    with torch.no_grad():
        for feat, label, idx in te_ld:
            p = torch.softmax(model(feat.to(device)),dim=1).cpu().numpy()
            for i, si in enumerate(idx.tolist()):
                row_ann = ann_test.iloc[si]
                all_probs.append({
                    'fold'         : fold,
                    'subject_id'   : row_ann['subject_id'],
                    'clip'         : row_ann['clip'],
                    'stage'        : row_ann['stage'],
                    'true_label'   : int(label[i].item()),
                    'pred_label'   : int(np.argmax(p[i])),
                    'prob_low'     : float(p[i][0]),
                    'prob_high'    : float(p[i][1]),
                    'correct'      : int(np.argmax(p[i])) == int(label[i].item()),
                    'anxiety_level': row_ann['anxiety_level'],
                })

    all_records.extend(all_probs)
    print(f'Fold {fold:02d} done | best_uar={best_uar:.4f} | stopped_ep={epoch}')

df_clips = pd.DataFrame(all_records)
print(f'\nTotal clip records: {len(df_clips)}')

Fold 00 done | best_uar=0.5604 | stopped_ep=75
Fold 01 done | best_uar=0.5505 | stopped_ep=99
Fold 02 done | best_uar=0.5533 | stopped_ep=81
Fold 03 done | best_uar=0.5838 | stopped_ep=83
Fold 04 done | best_uar=0.5714 | stopped_ep=99

Total clip records: 559


In [9]:
# ── Analisis 1: Konsistensi prediksi per subjek ───────────────────────────
print('=' * 55)
print('Analisis 1 — Konsistensi prediksi per subjek')
print('=' * 55)

subj_stats = []
for subj, grp in df_clips.groupby('subject_id'):
    n_clips   = len(grp)
    n_correct = grp['correct'].sum()
    true_lbl  = grp['true_label'].iloc[0]
    mean_prob = grp['prob_high'].mean()
    subj_pred = int(mean_prob >= 0.5)
    subj_stats.append({
        'subject_id'   : subj,
        'anxiety_level': grp['anxiety_level'].iloc[0],
        'true_label'   : true_lbl,
        'n_clips'      : n_clips,
        'n_correct'    : n_correct,
        'pct_correct'  : n_correct / n_clips,
        'subject_pred' : subj_pred,
        'subject_correct': subj_pred == true_lbl,
        'mean_prob_high': mean_prob,
    })

df_subj = pd.DataFrame(subj_stats)

print(f'Total subjek: {len(df_subj)}')
print(f'Subject-level accuracy: {df_subj["subject_correct"].mean():.4f}')
print(f'Subject UAR: {balanced_accuracy_score(df_subj["true_label"], df_subj["subject_pred"]):.4f}')
print()

# Distribusi konsistensi
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.01]
labels_b = ['0-20%','20-40%','40-60%','60-80%','80-100%']
df_subj['consistency_bin'] = pd.cut(df_subj['pct_correct'], bins=bins,
                                     labels=labels_b, right=False)
print('Distribusi % clip benar per subjek:')
print(df_subj['consistency_bin'].value_counts().sort_index().to_string())
print()

# Per anxiety level
print('Per anxiety level:')
print(df_subj.groupby('anxiety_level')['pct_correct'].agg(
    ['mean','std','min','max']).round(3).to_string())
print()

# Subjek yang subject-level benar tapi clip-level buruk
inconsistent = df_subj[(df_subj['subject_correct']) & (df_subj['pct_correct'] < 0.5)]
print(f'Subjek yang benar di subject-level tapi <50% clip benar: {len(inconsistent)}')
if len(inconsistent):
    print(inconsistent[['subject_id','anxiety_level','n_clips',
                          'n_correct','pct_correct','mean_prob_high']].to_string(index=False))

Analisis 1 — Konsistensi prediksi per subjek
Total subjek: 56
Subject-level accuracy: 0.4821
Subject UAR: 0.4792

Distribusi % clip benar per subjek:
consistency_bin
0-20%       0
20-40%      5
40-60%     18
60-80%     27
80-100%     6

Per anxiety level:
                mean    std  min  max
anxiety_level                        
high           0.559  0.141  0.3  0.8
low            0.602  0.172  0.3  1.0

Subjek yang benar di subject-level tapi <50% clip benar: 1
      subject_id anxiety_level  n_clips  n_correct  pct_correct  mean_prob_high
hizkia_elsadanta          high       10          4          0.4         0.54961


In [10]:
# ── Analisis 2: Akurasi per nomor clip dan per stage ─────────────────────
print('=' * 55)
print('Analisis 2 — Akurasi per clip number dan stage')
print('=' * 55)

print('Akurasi per clip number (1-5):')
clip_acc = df_clips.groupby('clip').apply(
    lambda g: balanced_accuracy_score(g['true_label'], g['pred_label'])
).round(4)
for c, v in clip_acc.items():
    bar = '█' * int(v * 30)
    print(f'  Clip {c}: {v:.4f}  {bar}')

print()
print('Akurasi per stage:')
stage_acc = df_clips.groupby('stage').apply(
    lambda g: balanced_accuracy_score(g['true_label'], g['pred_label'])
).round(4)
for s, v in stage_acc.items():
    bar = '█' * int(v * 30)
    print(f'  {s:8s}: {v:.4f}  {bar}')

print()
print('Akurasi per stage x anxiety_level:')
print(df_clips.groupby(['stage','anxiety_level']).apply(
    lambda g: balanced_accuracy_score(g['true_label'], g['pred_label'])
).round(4).to_string())

Analisis 2 — Akurasi per clip number dan stage
Akurasi per clip number (1-5):
  Clip q1: 0.6588  ███████████████████
  Clip q2: 0.6041  ██████████████████
  Clip q3: 0.5702  █████████████████
  Clip q4: 0.5408  ████████████████
  Clip q5: 0.5090  ███████████████

Akurasi per stage:
  after   : 0.6065  ██████████████████
  before  : 0.5552  ████████████████

Akurasi per stage x anxiety_level:
stage   anxiety_level
after   high             0.6095
        low              0.6034
before  high             0.5188
        low              0.5917


/tmp/ipykernel_47242/2022738527.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  clip_acc = df_clips.groupby('clip').apply(
/tmp/ipykernel_47242/2022738527.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stage_acc = df_clips.groupby('stage').apply(
/home/inadio/skripkir/pulse-live/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  w

In [11]:
# ── Analisis 3: Feature importance via permutation ────────────────────────
print('=' * 55)
print('Analisis 3 — Feature importance (permutation)')
print('=' * 55)

# Gunakan fold terakhir sebagai representasi
last_fold = folds[-1]
dataset.set_mode('eval'); dataset.clear_cache()
X_all, y_all, subs_all = dataset.get_all_features()
X_tr = X_all[last_fold['train_idx']]
y_tr = y_all[last_fold['train_idx']]
X_te = X_all[last_fold['test_idx']]
y_te = y_all[last_fold['test_idx']]
norm = Normalizer()
X_tr_n = norm.fit_transform(X_tr)
X_te_n = norm.transform(X_te)

# Wrapper sklearn untuk model torch
class TorchWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model, device):
        self.model  = model
        self.device = device
    def fit(self, X, y): return self
    def predict(self, X):
        self.model.eval()
        with torch.no_grad():
            out = self.model(torch.from_numpy(X.astype(np.float32)).to(self.device))
            return torch.argmax(out, dim=1).cpu().numpy()
    def score(self, X, y):
        return balanced_accuracy_score(y, self.predict(X))

# Load checkpoint fold terakhir
ckpt_files = sorted([f for f in os.listdir('./checkpoints')
                      if f.endswith('.pt')], reverse=True)
if ckpt_files:
    last_ckpt = torch.load(f'./checkpoints/{ckpt_files[0]}')
    model_pi  = SpatialTemporalCNN().to(device)
    model_pi.load_state_dict(last_ckpt)
    wrapper = TorchWrapper(model_pi, device)

    result = permutation_importance(
        wrapper, X_te_n, y_te,
        n_repeats=30, random_state=42,
        scoring='balanced_accuracy')

    # Nama fitur
    fe = FeatureExtractor()
    feat_names = []
    for k in range(fe.K_APEX):
        for r in fe.ROI_ORDER:
            for f in ['mean_mag','max_mag','net_dx','net_dy']:
                feat_names.append(f'apex{k+1}_{r}_{f}')
    for k in range(fe.K_APEX):
        for pair in ['eye_LR','eyebrow_LR']:
            for f in ['d_mag','d_dx','d_dy']:
                feat_names.append(f'apex{k+1}_{pair}_{f}')

    imp_df = pd.DataFrame({
        'feature'   : feat_names,
        'importance': result.importances_mean,
        'std'       : result.importances_std,
    }).sort_values('importance', ascending=False)

    print('Top 20 fitur (permutation importance):')
    print(imp_df.head(20).to_string(index=False))

    print(f'\nBottom 10 fitur (least important):')
    print(imp_df.tail(10).to_string(index=False))
else:
    print('Checkpoint tidak ditemukan. Jalankan train_groupkfold.ipynb dulu.')

Analisis 3 — Feature importance (permutation)
Top 20 fitur (permutation importance):
                    feature  importance      std
          apex1_eye_LR_d_dx    0.018869 0.013424
apex3_right_eyebrow_max_mag    0.016964 0.012039
  apex2_left_eyebrow_net_dx    0.015298 0.018687
   apex2_right_eye_mean_mag    0.011845 0.017094
          apex3_eye_LR_d_dy    0.010417 0.018752
          apex1_eye_LR_d_dy    0.008631 0.012432
    apex3_right_eye_max_mag    0.007857 0.019407
 apex2_left_eyebrow_max_mag    0.007619 0.016776
         apex3_lips_max_mag    0.007262 0.017743
      apex2_eyebrow_LR_d_dy    0.007143 0.019266
     apex2_left_eye_max_mag    0.005833 0.020239
      apex1_eyebrow_LR_d_dy    0.005655 0.017035
         apex1_lips_max_mag    0.005417 0.008267
          apex2_lips_net_dx    0.004048 0.011836
apex2_right_eyebrow_max_mag    0.003571 0.016463
     apex1_right_eye_net_dy    0.002500 0.019346
      apex3_left_eye_net_dy    0.001131 0.019635
     apex3_right_eye_net_dy   -0.

In [12]:
# ── Analisis 4: Confidence distribution ─────────────────────────────────
print('=' * 55)
print('Analisis 4 — Distribusi confidence prediksi')
print('=' * 55)

for lbl_name, lbl_val in [('HIGH', 1), ('LOW', 0)]:
    grp = df_clips[df_clips['true_label'] == lbl_val]
    correct = grp[grp['correct']]
    wrong   = grp[~grp['correct']]
    print(f'\n{lbl_name} ({len(grp)} clips):')
    print(f'  Benar  ({len(correct)}): prob_high mean={correct["prob_high"].mean():.3f} '
          f'std={correct["prob_high"].std():.3f}')
    print(f'  Salah  ({len(wrong)}): prob_high mean={wrong["prob_high"].mean():.3f} '
          f'std={wrong["prob_high"].std():.3f}')

# Clip yang paling tidak yakin (prob mendekati 0.5)
df_clips['confidence'] = (df_clips['prob_high'] - 0.5).abs()
uncertain = df_clips[df_clips['confidence'] < 0.1].sort_values('confidence')
print(f'\nClip dengan confidence sangat rendah (<0.1 dari 0.5): {len(uncertain)}')
print(f'  Dari total {len(df_clips)} clip = {len(uncertain)/len(df_clips)*100:.1f}%')

Analisis 4 — Distribusi confidence prediksi

HIGH (265 clips):
  Benar  (147): prob_high mean=0.654 std=0.115
  Salah  (118): prob_high mean=0.328 std=0.117

LOW (294 clips):
  Benar  (176): prob_high mean=0.361 std=0.110
  Salah  (118): prob_high mean=0.673 std=0.112

Clip dengan confidence sangat rendah (<0.1 dari 0.5): 217
  Dari total 559 clip = 38.8%
